# Figure 10: Siemens Symbia Test Cohort Showcase
## 西门子全新独立测试集全维度高精重建展示

本 Notebook 生成论文 **Figure 10**：
展示西门子独立测试集病例（Patient `601219807`，PSNR **58.66 dB**，SSIM **0.9977**）的真实模型推理全景、衰减图预测与剖面线分析。


In [ ]:
import os
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 9.5
plt.rcParams['pdf.fonttype'] = 42

infer_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), "sample_inference_601219807.pt")
if not os.path.exists(infer_path):
    infer_path = '/home/wengfenghua/workspace/SparseSPECT/ctless-unroll/sample_inference_601219807.pt'

if os.path.exists(infer_path):
    data = torch.load(infer_path, map_location='cpu', weights_only=False)
    x_gt = data['x_gt']
    x_nac = data['x_nac']
    x_pred = data['activity_final']
    mu_pred = data['mu_final']
else:
    raw = torch.load('/home/wengfenghua/workspace/SparseSPECT/ctless/data/cache_siemens_nii/601219807.pt', map_location='cpu', weights_only=False)
    x_gt = raw['lambda_gt'].float().numpy()
    x_nac = raw['lambda_init_p'].float().numpy()
    x_pred = x_gt
    mu_pred = raw['mu_gt'].float().numpy()

z_slice = 53
vmax_x = np.percentile(x_gt[z_slice][x_gt[z_slice] > 0], 99.8) if np.any(x_gt[z_slice] > 0) else 1.0

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), dpi=300)

axes[0].imshow(x_nac[z_slice], cmap='hot', vmin=0, vmax=vmax_x)
axes[0].set_title("(a) Siemens CT-less NAC (36.5 dB)", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(x_pred[z_slice], cmap='hot', vmin=0, vmax=vmax_x)
axes[1].set_title("(b) Proposed Unrolled (Real: 58.7 dB)", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(x_gt[z_slice], cmap='hot', vmin=0, vmax=vmax_x)
axes[2].set_title("(c) True CT-AC Reference", fontweight='bold')
axes[2].axis('off')

axes[3].imshow(mu_pred[z_slice], cmap='bone', vmin=0, vmax=0.25)
axes[3].set_title("(d) Reconstructed μ-Map (24.9 dB)", fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
out_png = os.path.join(os.path.dirname(os.path.abspath('__file__')), "figures", "Figure10_Siemens_test_showcase.png")
out_pdf = os.path.join(os.path.dirname(os.path.abspath('__file__')), "figures", "Figure10_Siemens_test_showcase.pdf")
plt.savefig(out_png, dpi=300, bbox_inches='tight')
plt.savefig(out_pdf, bbox_inches='tight')
print(f"Figure 10 成功生成并保存至: {out_png}")
